# Charlotte Project | Week 11 · Class 3
## Feature scaling, categorical encoding & leakage-safe preprocessing
**Interactive teaching notebook — not the lab solution**

In this lesson we prepare numeric and categorical business data for downstream machine learning. The examples use a small **synthetic coffee-shop dataset**, while the assessed lab uses a separate **public botanical dataset**. Both are intentionally different from oil-and-gas telemetry.

**What you should be able to do:** (1) explain why scale and representation matter, (2) implement standard and min–max scaling, (3) select appropriate category encodings, (4) split before fitting transformations, (5) use a `ColumnTransformer` pipeline, and (6) defend your decisions.

**Estimated classroom time:** 180 minutes. Formal work is completed in `.py` files, not here.

## A. Why do features need preparation? (25 minutes)
### Concept introduction
A *feature* is an input variable used in an analysis or model. Different sources represent features differently: transaction value might be measured in thousands of currency units, visit count in single digits, and membership level as a string. Many algorithms use distances or coefficient optimisation; the magnitude of features can therefore affect their behaviour. Algorithms based on trees often need less scaling, so **do not assume every model requires scaling**.

Categorical fields are labels, not quantities. A code such as `bronze=1, silver=2, gold=3` introduces a numerical ordering that may not be justified. A one-hot representation can instead create a separate indicator for each category.

**Guiding question:** If transaction value ranges from 5 to 5,000 and store visits from 1 to 10, which may dominate distance calculations without scaling?

In [11]:
import pandas as pd
import numpy as np
shop = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"],
    "spend_usd": [12., 40., 550., 25., 80., 62., 18., np.nan],
    "visits": [2, 4, 6, 3, 9, 5, 2, 7],
    "channel": ["app", "walk_in", "app", "web", "web", "app", None, "walk_in"],
    "joined": ["yes", "no", "no", "yes", "yes", "no", "yes", "no"]
})
print(shop.dtypes)
display(shop)

customer_id        str
spend_usd      float64
visits           int64
channel            str
joined             str
dtype: object


,customer_id,spend_usd,visits,channel,joined
0,C1,12.0,2,app,yes
1,C2,40.0,4,walk_in,no
2,C3,550.0,6,app,no
3,C4,25.0,3,web,yes
4,C5,80.0,9,web,yes
5,C6,62.0,5,app,no
6,C7,18.0,2,NaN,yes
7,C8,NaN,7,walk_in,no


### Practical interpretation
`customer_id` is a record identifier, not an explanatory feature. `joined` is the outcome we might eventually predict, so it must not enter the feature matrix. Missing fields should be addressed explicitly rather than silently treated as zero. **Common mistake:** scaling the outcome column together with predictors.

## B. Standardisation and min–max scaling (35 minutes)
### What they mean
**Standardisation** subtracts a feature's training mean and divides by its training standard deviation: `z = (x - mean) / std`. Its output is *not* guaranteed to be between -1 and 1. An extreme value can remain extreme. **Min–max scaling** uses `(x - train_min) / (train_max - train_min)` to map the observed *training* range to 0–1. New test values outside that range can map outside 0–1; this is expected.

Choose the transformation based on the downstream method and data. Scaling alone does **not** fix wrong units, missing measurements or erroneous records.

**Common mistakes:** (a) expecting standardised values to always be bounded, (b) fitting a new scaler on test data, (c) forgetting that outliers affect mean/std and min/max.

In [12]:
 
from sklearn.preprocessing import StandardScaler, MinMaxScaler
sample = pd.DataFrame({"spend_usd": [10., 30., 60., 100.], "visits": [1., 2., 4., 8.]})
std = StandardScaler().fit_transform(sample)
mm = MinMaxScaler().fit_transform(sample)
print("Standardised:\n", np.round(std, 3))
print("Min–max:\n", np.round(mm, 3))

Standardised:
 [[-1.18  -1.026]
 [-0.59  -0.653]
 [ 0.295  0.093]
 [ 1.474  1.585]]
Min–max:
 [[0.    0.   ]
 [0.222 0.143]
 [0.556 0.429]
 [1.    1.   ]]


In [13]:
scaler = MinMaxScaler().fit(sample)
new_customer = pd.DataFrame({"spend_usd": [120.], "visits": [10.]})
print("New values may exceed 1:", scaler.transform(new_customer))

New values may exceed 1: [[1.22222222 1.28571429]]


### Try it yourself
Change the training `spend_usd` from 100 to 1,000. Predict which scaling method changes the most for other observations, then re-run and explain. This is a formative classroom check, not an additional homework task.

## C. Encoding categorical features (35 minutes)
### Concept introduction
Categorical variables can be **nominal** (no inherent order; for example, sales channel) or **ordinal** (ordered; for example, low/medium/high risk). One-hot encoding is often a clear default for nominal variables because it does not invent distances between category labels. Ordinal encoding is appropriate only when the ordering is meaningful and documented; do not apply it mechanically to categories such as towns or departments.

With `handle_unknown="ignore"`, a category not observed during training will not crash transformation. The new category gets zeros across that feature's one-hot indicators, which is useful operationally but can hide unseen values. Monitor category drift in production.

**Common mistakes:** using integer labels for nominal categories, encoding a unique identifier, or allowing the target into the predictors.

In [14]:
from sklearn.preprocessing import OneHotEncoder
train_channels = pd.DataFrame({"channel": ["app", "web", "walk_in", "app"]})
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(train_channels)
print(encoder.get_feature_names_out())
print(encoded.astype(int))

['channel_app' 'channel_walk_in' 'channel_web']
[[1 0 0]
 [0 0 1]
 [0 1 0]
 [1 0 0]]


In [15]:
new_channels = pd.DataFrame({"channel": ["app", "partner_kiosk"]})
print("Unseen category (all zeros for that feature):\n", encoder.transform(new_channels))

Unseen category (all zeros for that feature):
 [[1. 0. 0.]
 [0. 0. 0.]]


### When not to one-hot everything
A column with thousands of unique identifiers creates thousands of sparse indicators, rarely appropriate for a beginner ML feature matrix. High-cardinality categoricals need deliberate treatment. For today's core assessment, categorical fields have only a few known levels.

## D. Train/test separation and data leakage (35 minutes)
### Concept introduction
A *transformer* estimates something from data during `fit`: a scaler learns the mean and standard deviation, an imputer learns the median or most common value, and an encoder learns category vocabulary. Fitting these on the entire dataset **before splitting** lets information from the future evaluation set influence feature preparation. This is data leakage even if the target column itself is not included.

**Correct sequence:** split first; fit the preprocessing pipeline on training features only; use the **same fitted pipeline** to transform both training and held-out test features. Note: stratification can use labels to preserve class ratios in the split; those labels are still not predictors.

**Common mistake:** `scaler.fit_transform(X)` then `train_test_split(...)`. This gives the scaler access to test-set statistics.

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
features = shop[["spend_usd", "visits", "channel"]]
labels = shop["joined"]
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.25, random_state=3)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

Training rows: 6 Test rows: 2


## E. Composing a safe preprocessing pipeline (35 minutes)
### Why pipelines matter
Pipelines combine steps in an explicit sequence. Here numeric values are imputed using the training median, then standardised. Categories are imputed using the most frequent training category, then one-hot encoded. A `ColumnTransformer` applies the right branch to the right columns. Each branch is fitted only on training data.

This is a **preprocessing exercise**, not a model accuracy contest. We evaluate successful, consistent transformation before introducing model training in later weeks.

In [17]:
num_cols = ["spend_usd", "visits"]
cat_cols = ["channel"]
numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
category_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", numeric_pipe, num_cols), ("cat", category_pipe, cat_cols)], sparse_threshold=0)
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Train and test shape:", X_train_ready.shape, X_test_ready.shape)
print("Feature names:", preprocessor.get_feature_names_out().tolist())

Train and test shape: (6, 5) (2, 5)
Feature names: ['num__spend_usd', 'num__visits', 'cat__channel_app', 'cat__channel_walk_in', 'cat__channel_web']


In [18]:
print("Finite train features:", np.isfinite(X_train_ready).all())
print("Finite test features:", np.isfinite(X_test_ready).all())
print("Train numeric means after scaling:", np.round(X_train_ready[:, :2].mean(axis=0), 5))

Finite train features: True
Finite test features: True
Train numeric means after scaling: [0. 0.]


### Self-check
Why should the mean of training numeric features be close to zero? Why might the held-out test mean *not* be exactly zero? What would happen if a new transaction has an unseen channel?

## F. Transfer to other formats, AWS and your lab (15 minutes)
### Format awareness
CSV and Excel often supply the same *tabular* features but differ in metadata and type inference. JSON may contain nested attributes that must be flattened. SQL queries can produce training tables; Parquet can store engineered columns efficiently. No additional graded task is assigned on these formats today: the notes include short reference examples.

### AWS DEA-C01 connection (foundational, not a cloud deployment lab)
For large production data pipelines, AWS Glue can perform managed ETL transformations and output datasets to S3. Today's `ColumnTransformer` is a *local Python/ML preprocessing tool*, not the same service as Glue, and is not itself an AWS exam objective. Distinguish data engineering transformations from downstream feature preprocessing.

**Scenario 1:** A team processes daily CSV drops into analytics-ready Parquet. Which AWS service would you consider for managed ETL, and why?
**Scenario 2:** A test environment has unseen region categories. Why should transformation logic be reproducible and versioned?
**Scenario 3:** Would you store plaintext cloud credentials in the GitHub repository? Why not?

### Independent lab handoff
Open `lab/Week11_Class3_Student_Lab_Guide.docx`. You will work with an independent public-derived botanical measurement dataset and add code to `preprocess_iris.py`. Commit in your personal repository and submit a PR. **Do not copy the notebook's toy dataset as your assessed output.**